# Chapter 18 — Resampling: Bootstrap & Jackknife
*Track 3: Data Scientists*

## 🎯 Learning Objectives
- Understand bootstrapping for confidence intervals
- Implement the jackknife for bias estimation
- Apply resampling to ML model uncertainty

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. Concept Review — Resampling Without Assumptions

The **bootstrap** simulates the sampling distribution by resampling with
replacement from the observed data:

1. Draw $B$ bootstrap samples of size $n$ (with replacement)
2. Compute the statistic $\hat\theta^*_b$ for each
3. The distribution of $\hat\theta^*_b$ estimates the sampling distribution

No Gaussian assumptions required — works for **any** statistic.

In [ ]:
# Bootstrap CI for median — no closed-form formula exists
data = rng.exponential(scale=3, size=50)
B = 2000
boot_medians = [rng.choice(data, size=len(data), replace=True).mean() for _ in range(B)]

ci_lo, ci_hi = np.percentile(boot_medians, [2.5, 97.5])
print(f"Sample mean: {data.mean():.3f}")
print(f"Bootstrap 95% CI for mean: [{ci_lo:.3f}, {ci_hi:.3f}]")
print(f"Analytic CI (t-dist): [{data.mean()-1.96*data.std()/np.sqrt(len(data)):.3f}, {data.mean()+1.96*data.std()/np.sqrt(len(data)):.3f}]")

plt.hist(boot_medians, bins=50, density=True, alpha=0.7)
plt.axvspan(ci_lo, ci_hi, alpha=0.3, color="red", label="95% CI")
plt.axvline(data.mean(), color="black", linestyle="--", label="Sample mean")
plt.title("Bootstrap Distribution of Sample Mean")
plt.legend(); plt.tight_layout(); plt.show()

## 2. Math Walkthrough — Jackknife Bias Estimation

The **jackknife** removes one observation at a time:
$$\hat\theta_{(-i)} = \hat\theta \text{ computed without } x_i$$

Jackknife bias estimate:
$$\widehat{\text{bias}} = (n-1)(\bar{\hat\theta}_{(-\cdot)} - \hat\theta)$$

Jackknife variance estimate:
$$\widehat{\text{Var}}(\hat\theta) = \frac{n-1}{n}\sum_{i=1}^n (\hat\theta_{(-i)} - \bar{\hat\theta}_{(-\cdot)})^2$$

In [ ]:
# Jackknife for variance of correlation coefficient
np.random.seed(0)
n = 40
x = rng.normal(0, 1, n)
y = 0.7*x + rng.normal(0, 0.7, n)
r_hat = np.corrcoef(x, y)[0, 1]

jack_rs = [np.corrcoef(np.delete(x, i), np.delete(y, i))[0,1] for i in range(n)]
jack_mean = np.mean(jack_rs)
jack_bias = (n-1) * (jack_mean - r_hat)
jack_var  = (n-1)/n * sum((r - jack_mean)**2 for r in jack_rs)

print(f"r_hat = {r_hat:.4f}")
print(f"Jackknife bias = {jack_bias:.4f}")
print(f"Jackknife SE   = {np.sqrt(jack_var):.4f}")

# Bootstrap for comparison
boot_rs = [np.corrcoef(*rng.choice(np.c_[x,y], n, replace=True).T)[0,1] for _ in range(2000)]
print(f"Bootstrap SE   = {np.std(boot_rs):.4f}")

## 3–6. Bootstrap for ML, Real Dataset, Practice

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.datasets import load_diabetes

diabetes = load_diabetes()
X_d, y_d = diabetes.data, diabetes.target

# Bootstrap coefficients
lr = LinearRegression()
coef_boots = []
for _ in range(1000):
    idx = rng.choice(len(X_d), len(X_d), replace=True)
    lr.fit(X_d[idx], y_d[idx])
    coef_boots.append(lr.coef_.copy())
coef_boots = np.array(coef_boots)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
feature_names = diabetes.feature_names
for i, ax in enumerate(axes.flat):
    if i < X_d.shape[1]:
        ax.hist(coef_boots[:, i], bins=30, alpha=0.7)
        ci = np.percentile(coef_boots[:, i], [2.5, 97.5])
        ax.axvline(ci[0], color="red", linestyle="--", lw=1)
        ax.axvline(ci[1], color="red", linestyle="--", lw=1)
        ax.axvline(0, color="black", lw=0.8)
        ax.set_title(feature_names[i], fontsize=9)
plt.suptitle("Bootstrap Distribution of Linear Regression Coefficients", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Practice: bootstrap CI for Sharpe ratio
returns = rng.normal(0.001, 0.02, 252)  # daily returns for 1 year
sharpe = returns.mean() / returns.std() * np.sqrt(252)
boot_sharpes = [(rng.choice(returns, len(returns), replace=True).mean() /
                 rng.choice(returns, len(returns), replace=True).std() * np.sqrt(252))
                for _ in range(2000)]
lo, hi = np.percentile(boot_sharpes, [2.5, 97.5])
print(f"Sharpe ratio: {sharpe:.3f}")
print(f"Bootstrap 95% CI: [{lo:.3f}, {hi:.3f}]")
print(f"Significantly > 0: {lo > 0}")